In [ ]:
import numpy as np
import random
import matplotlib.pyplot as plt

In [ ]:
# 1. Train first
split="train"
dataset_path = f"/mnt/home/lserrano/disco-ball/datasets/{split}.npz"

In [ ]:
data = np.load(dataset_path)
trajectories = data["trajectories"]  # shape: [N, T, H]
velocities = data["velocities"]
diffusivities = data["diffusivities"]

In [ ]:
N, T, H = trajectories.shape
indices = np.random.randint(0, N, size=3)

for idx in indices:
    traj = trajectories[idx]  # shape: [T, H]
    velocity = velocities[idx]
    diffusivity = diffusivities[idx]
    plt.figure(figsize=(10, 6))
    # Plot as an image (time x space)
    plt.imshow(traj, aspect="auto", origin="lower", cmap="viridis")
    plt.colorbar(label="u")
    plt.xlabel("Space")
    plt.ylabel("Time")
    plt.title(f"{split} idx={idx} | v={velocity:.3g} | D={diffusivity:.3g}")
    plt.tight_layout()
    #plt.savefig(f"{plot_dir}/{dataset_name}_sample{idx}.png")
    plt.show()
    plt.close()
    # Also plot as line plots for a few time steps
    plt.figure(figsize=(10, 6))
    for t in np.linspace(0, T-1, min(8, T), dtype=int):
        plt.plot(traj[t], label=f"t={t}")
    plt.xlabel("Space")
    plt.ylabel("u")
    plt.title(f"{split} idx={idx} | v={velocity:.3g} | D={diffusivity:.3g} (lines)")
    plt.legend()
    plt.tight_layout()
    plt.show()
    #plt.savefig(f"{plot_dir}/{dataset_name}_sample{idx}_lines.png")
    plt.close()

In [ ]:
#### with finite differences

In [ ]:
# Example Usage (assuming 1D simulation of a sine wave):
if __name__ == '__main__':
    # Simulation parameters
    num_points = 256  # Spatial grid points
    domain_length = 2.0 * torch.pi # Spatial domain (e.g., 0 to 2pi)
    dx = domain_length / num_points # Spatial step size
    dt = 0.01  # Time step size for each operator application

    # Operator parameters
    velocity = (0.5,) # Advection velocity for 1D
    diffusivity = 0.01 # Diffusion coefficient

    # Initialize the operator
    ad_operator = AdvectionDiffusion(velocity=velocity, diffusivity=diffusivity, dx=dx, dt=dt)

    # Initial condition (a sine wave)
    x = torch.linspace(0, domain_length, num_points, dtype=torch.float32)
    # u_initial shape: (batch_size, channels, spatial_dim1, ...)
    u = torch.sin(x).unsqueeze(0).unsqueeze(0) # -> (1, 1, num_points) for batch=1, channel=1

    # Simulate for a total time
    total_simulation_time = 10.0
    num_steps = int(total_simulation_time / dt)

    print(f"Starting simulation for {num_steps} steps...")

    # Store results for visualization (e.g., every 100 steps)
    simulation_results = [u.squeeze().numpy()] # Store initial state (numpy for plotting)

    for step in range(num_steps):
        u = ad_operator(u) # Apply one time step of advection and diffusion
        
        if (step + 1) % 100 == 0 or step == num_steps - 1:
            simulation_results.append(u.squeeze().numpy())
            print(f"Step {step+1}/{num_steps} completed.")

    print("Simulation finished.")

    # Basic plotting using matplotlib (if available)
    try:
        import matplotlib.pyplot as plt
        
        plt.figure(figsize=(10, 6))
        plt.title('Advection-Diffusion Simulation (Finite Difference Advection)')
        plt.xlabel('Space')
        plt.ylabel('Amplitude')
        
        # Plot initial state
        plt.plot(x.numpy(), simulation_results[0], label='Initial (t=0)', linestyle='--')
        
        # Plot intermediate and final states
        time_per_step = dt * 100 # Time elapsed for each plotted line (if plotting every 100 steps)
        for i, data in enumerate(simulation_results[1:]):
            current_time = (i * time_per_step) + dt # Adjust to actual time of stored data
            plt.plot(x.numpy(), data, label=f't={current_time:.2f}')
        
        plt.legend()
        plt.grid(True)
        plt.show()

    except ImportError:
        print("\nMatplotlib not found. Install it to visualize the results: pip install matplotlib")
        print("Final state (first 10 values):", u.squeeze().numpy()[:10])

In [ ]:
from typing import Tuple, Union

class AdvectionDiffusion:
    """
    Implements the Advection-Diffusion operator using a spectral (Fourier) method
    for both advection and diffusion components.
    
    Advection in Fourier space involves multiplication by exp(-i * (k . v) * t).
    Diffusion in Fourier space involves multiplication by exp(-D * |k|^2 * t).
    These operations are combined into a single spectral filter.
    """

    def __init__(self, velocity: Tuple, diffusivity: float):
        """
        Initializes the AdvectionDiffusion operator.

        Args:
            velocity (Tuple): A tuple representing the velocity vector (e.g., (vx,) for 1D, (vx, vy) for 2D).
            diffusivity (float): The diffusion coefficient.
        """
        self.velocity = velocity
        self.diffusivity = diffusivity
    
    def __call__(self, t: torch.Tensor, u: torch.Tensor, seed: Union[int, None] = None) -> torch.Tensor:
        """
        Applies the advection-diffusion operator to a field 'u' over time steps 't'.

        Args:
            t (torch.Tensor): A 1D tensor of time steps for which to compute the solution.
                              Shape: (num_timesteps,).
            u (torch.Tensor): The initial field tensor.
                              Expected shape: (batch, channels, spatial_dim1) for 1D,
                              or (batch, channels, spatial_dim1, spatial_dim2) for 2D.
            seed (int | None): An optional seed for randomness (not used in this deterministic spectral method).

        Returns:
            torch.Tensor: The advected-diffused field at each time step.
                          Shape: (batch, num_timesteps, channels, spatial_dim1, [spatial_dim2]).
        
        Raises:
            ValueError: If the number of spatial dimensions is not 1 or 2.
        """
        
        self._input_checks(t, u)

        # Determine the number of spatial dimensions (e.g., 1 for 1D, 2 for 2D)
        # u should be (batch, channels, H, [W])
        d = u.ndim - 2 

        if d == 1: # 1D case: u has shape (batch, channels, H)
            H = u.shape[-1]
            
            # Generate 1D wavenumbers for the real-to-complex FFT (rfftn)
            # k_x will have shape (H//2 + 1,)
            k_x = 2 * torch.pi * torch.fft.rfftfreq(H, d=1.0, device=u.device, dtype=u.dtype)

            # Compute the squared magnitude of the wavenumber vector (K^2) for diffusion
            # Shape: (H//2 + 1,)
            k_sq = k_x**2
            
            # Compute the dot product of the wavenumber vector and velocity vector (K . V) for advection
            # Assumes velocity is (vx,) for 1D
            # Shape: (H//2 + 1,)
            k_dot_v = k_x * self.velocity[0]

            # Reshape k_sq and k_dot_v to allow broadcasting with u_f later.
            # They need to match the last 'd' dimensions of u_f, and have 1s for batch/channel.
            # For 1D, u_f is (batch, channels, k_H). k_sq/k_dot_v should be (1, 1, k_H).
            k_sq = k_sq.view(1, 1, -1)
            k_dot_v = k_dot_v.view(1, 1, -1)

        elif d == 2: # 2D case: u has shape (batch, channels, H, W)
            H, W = u.shape[-2:]

            # Generate 2D wavenumbers.
            # For rfftn, the first spatial dimension (H) uses fftfreq (full spectrum).
            # The last spatial dimension (W) uses rfftfreq (half spectrum for real input).
            k_x = 2 * torch.pi * torch.fft.fftfreq(H, d=1.0, device=u.device, dtype=u.dtype)
            k_y = 2 * torch.pi * torch.fft.rfftfreq(W, d=1.0, device=u.device, dtype=u.dtype)

            # Create a meshgrid of 2D wavenumbers
            # k_x_grid shape: (H, 1), k_y_grid shape: (1, W//2 + 1)
            k_x_grid, k_y_grid = torch.meshgrid(k_x, k_y, indexing='ij')

            # Compute the squared magnitude of the wavenumber vector (K^2) for diffusion
            # Shape: (H, W//2 + 1)
            k_sq = k_x_grid**2 + k_y_grid**2
            
            # Compute the dot product of the wavenumber vector and velocity vector (K . V) for advection
            # Assumes velocity is (vx, vy) for 2D
            # Shape: (H, W//2 + 1)
            k_dot_v = k_x_grid * self.velocity[0] + k_y_grid * self.velocity[1]

            # Reshape k_sq and k_dot_v for broadcasting.
            # For 2D, u_f is (batch, channels, k_H, k_W). k_sq/k_dot_v should be (1, 1, k_H, k_W).
            k_sq = k_sq.view(1, 1, H, W//2 + 1)
            k_dot_v = k_dot_v.view(1, 1, H, W//2 + 1)
        else:
            raise ValueError(f"Unsupported number of spatial dimensions: {d}. Only 1D or 2D are supported.")

        # Define the dimensions over which to perform the FFT
        # For 1D: (-1,)
        # For 2D: (-2, -1)
        dims_fft = tuple(range(-d, 0))

        # Apply Real-to-Complex FFT to the input field 'u' along the spatial dimensions
        # u_f will be a complex tensor.
        # Shape: (batch, channels, k_H, [k_W])
        u_f = torch.fft.rfftn(u, dim=dims_fft)

        # Prepare the time tensor 't' for broadcasting with the spectral filter.
        # 't' has shape (num_timesteps,). We need it to be broadcastable with k_sq/k_dot_v,
        # which are (1, 1, k_H, [k_W]).
        # So, t_exp needs to have shape (num_timesteps, 1, 1, *([1]*d)).
        t_exp = t.view(-1, *([1] * k_sq.ndim)) # e.g., for 2D, k_sq.ndim is 4, so t_exp is (num_timesteps, 1, 1, 1, 1)

        # Compute the combined spectral filter for advection and diffusion for all time steps.
        # The filter will be complex. '1j' is represented by torch.complex(0.0, 1.0).
        # Shape: (num_timesteps, 1, 1, k_H, [k_W])
        spectral_filter = torch.exp(
            -torch.complex(0.0, 1.0) * k_dot_v * t_exp
            - self.diffusivity * k_sq * t_exp
        )

        # Apply the spectral filter to the Fourier transformed field 'u_f'.
        # To enable broadcasting for the time dimension, we expand u_f.
        # u_f_expanded shape: (batch, 1, channels, k_H, [k_W])
        u_f_expanded = u_f.unsqueeze(1) 

        # Perform the element-wise multiplication in Fourier space.
        # u_f_expanded shape: (batch, 1, channels, k_H, k_W)
        # spectral_filter shape: (num_timesteps, 1, 1, k_H, k_W)
        # Resulting shape: (batch, num_timesteps, channels, k_H, k_W)
        u_f_advected_diffused = u_f_expanded * spectral_filter

        # Define the dimensions over which to perform the Inverse FFT
        dims_ifft = dims_fft # Same spatial dimensions as FFT

        # Specify the output sizes for the Inverse FFT to ensure the correct spatial resolution.
        # This is important when using rfftn/irfftn to avoid truncation/padding issues.
        s_ifft = u.shape[-d:] # (H,) for 1D, (H, W) for 2D

        # Apply Inverse Real-to-Complex FFT to get the field back in the spatial domain.
        # The output 'ut' will have shape: (batch, num_timesteps, channels, H, [W])
        ut = torch.fft.irfftn(u_f_advected_diffused, s=s_ifft, dim=dims_ifft)
        
        return ut

In [ ]:
# --- Test Script ---
if __name__ == "__main__":
    # Parameters for 1D test case
    N = 256  # Spatial grid points
    L = 2 * torch.pi # Spatial domain length
    x = torch.linspace(0, L, N, endpoint=False, dtype=torch.float32)

    initial_velocity = (1.0,)  # Advection velocity
    initial_diffusivity = 0.1 # Diffusion coefficient
    
    # Time steps
    num_timesteps = 5
    max_time = 2.0
    t_values = torch.linspace(0, max_time, num_timesteps + 1, dtype=torch.float32)
    # We will use t_values[1:] for the actual calculation as t=0 is the initial state

    # Initial condition: A Gaussian pulse
    center = L / 4
    sigma_initial = 0.5
    u_initial = torch.exp(-((x - center)**2) / (2 * sigma_initial**2))

    # Reshape u_initial to (batch, channels, H)
    # Here, batch=1, channels=1
    u_initial_tensor = u_initial.view(1, 1, N)

    # Instantiate the AdvectionDiffusion operator
    advection_diffusion_op = AdvectionDiffusion(velocity=initial_velocity, diffusivity=initial_diffusivity)

    print(f"Initial field shape: {u_initial_tensor.shape}")
    print(f"Time steps: {t_values}")

    # Apply the operator
    # The operator expects t starting from time 0, but will return the solution at
    # those specific time points. So, we pass all t_values.
    # The output will have shape (batch, num_timesteps_passed, channels, H)
    ut_result = advection_diffusion_op(t=t_values, u=u_initial_tensor)

    print(f"Resulting field shape: {ut_result.shape}")

    # Plotting the results
    plt.figure(figsize=(10, 6))
    plt.plot(x.numpy(), u_initial.numpy(), label=f'Initial (t={t_values[0].item():.2f})', linestyle='--')

    # Plot the states at each time step
    for i in range(1, num_timesteps + 1):
        # Extract the field for the current time step.
        # ut_result has shape (batch, num_timesteps_passed, channels, H)
        # We need to get the field at time t_values[i], which is at index i in the second dimension.
        # Then squeeze to get (H,) for plotting.
        field_at_t = ut_result[0, i, 0, :].numpy()
        plt.plot(x.numpy(), field_at_t, label=f't={t_values[i].item():.2f}')

    plt.title('1D Advection-Diffusion using Spectral Method')
    plt.xlabel('Spatial Coordinate (x)')
    plt.ylabel('Amplitude (u)')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    print("\n--- Testing 2D Advection-Diffusion ---")

    # Parameters for 2D test case
    N_2d = 64 # Spatial grid points for 2D (NxN grid)
    L_2d = 2 * torch.pi
    x_2d = torch.linspace(0, L_2d, N_2d, endpoint=False, dtype=torch.float32)
    y_2d = torch.linspace(0, L_2d, N_2d, endpoint=False, dtype=torch.float32)
    X_2d, Y_2d = torch.meshgrid(x_2d, y_2d, indexing='ij')

    velocity_2d = (0.5, 0.7) # Advection velocity for 2D
    diffusivity_2d = 0.05 # Diffusion coefficient for 2D

    # Time steps for 2D
    num_timesteps_2d = 3
    max_time_2d = 1.0
    t_values_2d = torch.linspace(0, max_time_2d, num_timesteps_2d + 1, dtype=torch.float32)

    # Initial condition for 2D: A 2D Gaussian pulse
    center_x_2d, center_y_2d = L_2d / 4, L_2d / 4
    sigma_initial_2d = 0.3
    u_initial_2d = torch.exp(
        -((X_2d - center_x_2d)**2 + (Y_2d - center_y_2d)**2) / (2 * sigma_initial_2d**2)
    )

    # Reshape u_initial_2d to (batch, channels, H, W)
    u_initial_tensor_2d = u_initial_2d.view(1, 1, N_2d, N_2d)

    # Instantiate the 2D AdvectionDiffusion operator
    advection_diffusion_op_2d = AdvectionDiffusion(velocity=velocity_2d, diffusivity=diffusivity_2d)

    print(f"Initial 2D field shape: {u_initial_tensor_2d.shape}")
    print(f"2D Time steps: {t_values_2d}")

    # Apply the operator for 2D
    ut_result_2d = advection_diffusion_op_2d(t=t_values_2d, u=u_initial_tensor_2d)

    print(f"Resulting 2D field shape: {ut_result_2d.shape}")

    # Plotting 2D results (initial and final state)
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))

    # Initial state
    im0 = axes[0].imshow(u_initial_2d.numpy(), origin='lower', extent=[0, L_2d, 0, L_2d], cmap='viridis')
    axes[0].set_title(f'Initial 2D Field (t={t_values_2d[0].item():.2f})')
    axes[0].set_xlabel('X')
    axes[0].set_ylabel('Y')
    fig.colorbar(im0, ax=axes[0])

    # Final state
    # Extract the field at the last time step
    field_at_final_t_2d = ut_result_2d[0, -1, 0, :, :].numpy()
    im1 = axes[1].imshow(field_at_final_t_2d, origin='lower', extent=[0, L_2d, 0, L_2d], cmap='viridis')
    axes[1].set_title(f'Final 2D Field (t={t_values_2d[-1].item():.2f})')
    axes[1].set_xlabel('X')
    axes[1].set_ylabel('Y')
    fig.colorbar(im1, ax=axes[1])

    plt.suptitle('2D Advection-Diffusion using Spectral Method')
    plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to prevent title overlap
    plt.show()